# 05. Запасы, риски и бизнес-выводы

Цель ноутбука: рассчитать safety stock, reorder point и сценарные зоны stockout / overstock risk для товаров.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, RESULTS_DIR
from src.inventory import classify_stock_risk, scenario_inventory_metrics

In [ ]:
daily_sales = pd.read_parquet(PROCESSED_DATA_DIR / 'daily_sales.parquet')
inventory = scenario_inventory_metrics(
    daily_sales,
    group_columns=['stock_code', 'country'],
    lead_time_days=7,
    service_level=0.95,
)
inventory.head()

In [ ]:
# Сценарий без фактических остатков: проверяем чувствительность при разных условных остатках.
low_stock = inventory.copy()
low_stock['scenario_name'] = 'низкий остаток'
low_stock['stock_on_hand'] = low_stock['reorder_point'] * 0.7

planned_stock = inventory.copy()
planned_stock['scenario_name'] = 'плановый остаток'
planned_stock['stock_on_hand'] = planned_stock['reorder_point'] * 1.2

high_stock = inventory.copy()
high_stock['scenario_name'] = 'высокий остаток'
high_stock['stock_on_hand'] = high_stock['reorder_point'] * 2.5

scenario = pd.concat([low_stock, planned_stock, high_stock], ignore_index=True)
scenario['risk_type'] = classify_stock_risk(scenario)
scenario.head()

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
scenario.to_parquet(RESULTS_DIR / 'inventory_scenario.parquet', index=False)
scenario.groupby('risk_type', as_index=False).agg(sku_count=('stock_code', 'count'))

## Бизнес-выводы после запуска

- товары с высоким reorder point: `[A]`;
- товары с высоким safety stock: `[B]`;
- сценарный stockout risk: `[C]`;
- сценарный overstock risk: `[D]`;
- что предложить бизнесу: `[E]`.